In [ ]:
%pip install nfl_data_py

In [ ]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np

In [ ]:
# Load data for 2005-2024 (up to available data)
years = list(range(2005, 2025))
dfs = {}
for year in years:
    try:
        seasonal_data = nfl.import_seasonal_data(years=[year], s_type='REG')
        rosters = nfl.import_seasonal_rosters(years=[year])
        merged = pd.merge(seasonal_data, rosters, on='player_id', how='left')
        merged['ppg'] = merged['fantasy_points_ppr'] / merged['games']
        dfs[year] = merged
        print(f"Loaded {year} data")
    except Exception as e:
        print(f"Error loading {year}: {e}")

# Filter to RBs and select relevant columns
rb_features = []
for year, df in dfs.items():
    rb_df = df[df['position'] == 'RB'].copy()
    
    # Load pbp data for accurate team volumes
    try:
        pbp = nfl.import_pbp_data(years=[year], downcast=True, cache=False)
        pbp_reg = pbp[pbp['season_type'] == 'REG']
        
        # Aggregate team-level passing volumes from pbp
        team_volumes = pbp_reg.groupby('posteam').agg({
            'receiver_player_id': lambda x: x.notna().sum(),  # total targets
            'air_yards': 'sum',
            'yards_after_catch': 'sum'
        }).reset_index()
        team_volumes.columns = ['team', 'total_targets', 'total_air_yards', 'total_yac']
        
        # Get games per team from seasonal data
        team_games = df.groupby('team')['games'].first().reset_index()
        team_games.columns = ['team', 'games']
        
        # Merge
        team_volumes = team_volumes.merge(team_games, on='team', how='left')
        
        # Calculate per-game volumes
        team_volumes['targets_per_game'] = team_volumes['total_targets'] / team_volumes['games']
        team_volumes['air_yards_per_game'] = team_volumes['total_air_yards'] / team_volumes['games']
        team_volumes['yac_per_game'] = team_volumes['total_yac'] / team_volumes['games']
        
    except Exception as e:
        print(f"Error loading pbp for {year}: {e}")
        # Fallback to seasonal aggregation if pbp fails
        team_volumes = df.groupby('team').agg({
            'targets': 'sum',
            'passing_air_yards': 'sum',
            'receiving_yards_after_catch': 'sum',
            'games': 'first'
        }).reset_index()
        team_volumes.columns = ['team', 'total_targets', 'total_air_yards', 'total_yac', 'games']
        team_volumes['targets_per_game'] = team_volumes['total_targets'] / team_volumes['games']
        team_volumes['air_yards_per_game'] = team_volumes['total_air_yards'] / team_volumes['games']
        team_volumes['yac_per_game'] = team_volumes['total_yac'] / team_volumes['games']
    
    # Merge team volumes to rb_df
    rb_df = rb_df.merge(team_volumes[['team', 'targets_per_game', 'air_yards_per_game', 'yac_per_game']], on='team', how='left', suffixes=('', '_team'))
    
    # Calculate per-game stats
    per_game_cols = [
        'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_epa', 'rushing_first_downs', 'rushing_2pt_conversions',
        'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
        'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'targets_per_game', 'air_yards_per_game',
        'yac_per_game'
    ]
    
    for col in per_game_cols:
        if col in rb_df.columns:
            rb_df[f'{col}_per_game'] = rb_df[col] / rb_df['games']
    
    # Select final features
    feature_cols = [
        'player_id', 'player_name', 'season_year', 'games', 'age', 'draft_number', 'rookie_year', 'height', 'weight'
    ] + [f'{col}_per_game' for col in per_game_cols if f'{col}_per_game' in rb_df.columns] + ['ppg']
    
    rb_df['season_year'] = year
    rb_features.append(rb_df[feature_cols].dropna(subset=['games', 'age', 'draft_number', 'rookie_year']))

# Combine all years
rb_data = pd.concat(rb_features, ignore_index=True)

# Add experience level
rb_data['experience'] = rb_data['season_year'] - rb_data['rookie_year'] + 1

# Add next year's PPG as target (shift by player)
rb_data = rb_data.sort_values(['player_id', 'season_year'])
rb_data['ppg_next_year'] = rb_data.groupby('player_id')['ppg'].shift(-1)

# Filter to players with next year data and minimum games
model_data = rb_data[(rb_data['games'] >= 6) & (rb_data['ppg_next_year'].notna())].copy()

# Fill missing values with 0 for per-game stats (rookies, etc.)
per_game_features = [col for col in model_data.columns if '_per_game' in col]
model_data[per_game_features] = model_data[per_game_features].fillna(0)

print(f"Prepared {len(model_data)} RB seasons for modeling")
print("Features:", [col for col in model_data.columns if col not in ['player_id', 'player_name', 'season_year', 'ppg_next_year']])
print("Target: ppg_next_year")

In [ ]:
from sklearn.model_selection import train_test_split

# Extract features and target
# Exclude: ppg (duplicate), fantasy_points_ppr_per_game (derived), and keep rushing/receiving TDs separate
feature_cols = [col for col in model_data.columns 
                if col not in ['player_id', 'player_name', 'season_year', 'ppg_next_year', 'ppg', 'fantasy_points_ppr_per_game']]

X = model_data[feature_cols]
y = model_data['ppg_next_year']

# 85-15 random train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Features: {len(feature_cols)}")
print(f"\n2025 data: NOT AVAILABLE (HTTP 404 - season not complete)")

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Convert all columns to numeric and handle inf/nan values
X_train_numeric = X_train.astype(float)
X_test_numeric = X_test.astype(float)

# Replace inf with large values and fill NaN with 0
X_train_numeric = X_train_numeric.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test_numeric = X_test_numeric.replace([np.inf, -np.inf], np.nan).fillna(0)

# Train XGBoost model
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

xgb_model.fit(X_train_numeric, y_train, verbose=False)

# Predictions
y_train_pred = xgb_model.predict(X_train_numeric)
y_test_pred = xgb_model.predict(X_test_numeric)

# Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("XGBoost Model Performance")
print("=" * 50)
print(f"Train MAE:  {train_mae:.3f} PPG")
print(f"Test MAE:   {test_mae:.3f} PPG")
print(f"Test RMSE:  {test_rmse:.3f} PPG")
print(f"Test R²:    {test_r2:.3f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print("=" * 50)
print(feature_importance.head(29).to_string(index=False))